In [2]:

import pandas as pd
import numpy as np
import glob
import os
import shutil
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
from soja_clima.inmet import copiar_um_arquivo, agregar_diario
from soja_clima.inmet import COLUNAS_CONHECIDAS, COLUNAS_ESSENCIAIS

c:\Users\okubo\OneDrive\Documentos\PCD2\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
# ============================================================
# CÉLULA 1 — Imports e caminhos
# ============================================================
from pathlib import Path

import pandas as pd
import numpy as np
import glob
from tqdm.auto import tqdm

from soja_clima.inmet import ler_arquivo_inmet, agregar_diario

# Assume que o notebook roda de dentro de notebooks/
RAIZ_PROJETO = Path.cwd().parent
PASTA_RAIZ = RAIZ_PROJETO / "data" / "raw" / "DadosClimaticos"
PASTA_PROCESSADA = RAIZ_PROJETO / "data" / "processed"
PASTA_PROCESSADA.mkdir(parents=True, exist_ok=True)


# ============================================================
# CÉLULA 2 — Listar os arquivos (sem cópia nenhuma -- já é local)
# ============================================================
# set() é necessário mesmo com um único glob "case-insensitive"-safe:
# em sistemas de arquivo que não diferenciam maiúscula/minúscula
# (Windows, macOS padrão), os padrões *.CSV e *.csv casam com o MESMO
# arquivo duas vezes -- sem o set(), o total sai dobrado.
arquivos = sorted(set(
    glob.glob(f"{PASTA_RAIZ}/**/*.CSV", recursive=True) +
    glob.glob(f"{PASTA_RAIZ}/**/*.csv", recursive=True)
))
print(f"{len(arquivos)} arquivos encontrados em {PASTA_RAIZ}")

if not arquivos:
    raise RuntimeError(
        f"Nenhum arquivo encontrado em {PASTA_RAIZ}. Confira o caminho e se "
        "está rodando o Jupyter a partir de notebooks/."
    )


# ============================================================
# CÉLULA 3 — Agregação hora -> dia, arquivo por arquivo (streaming)
# ============================================================
# Nunca concatenamos os dados horários de todos os arquivos de uma vez
# (seriam dezenas de milhões de linhas). Cada arquivo é processado e
# reduzido para diário individualmente; só os diários são acumulados.
diarios = []
erros_processamento = []
for path in tqdm(arquivos, desc="Processando"):
    try:
        df_horario = ler_arquivo_inmet(path)
        diarios.append(agregar_diario(df_horario))
    except Exception as e:
        erros_processamento.append((path, str(e)))

print(f"\n{len(erros_processamento)} arquivo(s) com falha no processamento.")
for path, msg in erros_processamento[:20]:
    print(f"  - {path}: {msg}")

if not diarios:
    raise RuntimeError("Nenhum arquivo foi processado com sucesso — confira os erros acima.")

diario = pd.concat(diarios, ignore_index=True)
diario['amplitude_termica'] = diario['temp_max'] - diario['temp_min']
diario['dia_seco'] = diario['prec_total'] < 1
diario['gdd'] = ((diario['temp_max'] + diario['temp_min']) / 2 - 10).clip(lower=0)

# Descarta dias com muitas horas faltando (controle de qualidade)
diario = diario[diario['n_horas_validas'] >= 18].copy()

print("\nShape final (diário), antes das exclusões manuais:", diario.shape)


# ============================================================
# CÉLULA 4 — Aplicar exclusões decididas no 01_eda_horaria.ipynb
# ============================================================
caminho_exclusoes = PASTA_PROCESSADA / "exclusoes_manuais.csv"

if caminho_exclusoes.exists():
    exclusoes = pd.read_csv(caminho_exclusoes)
    exclusoes = exclusoes.dropna(subset=['codigo_estacao'])  # ignora linhas de exemplo vazias

    for _, r in exclusoes.iterrows():
        mask = (
            (diario['codigo_estacao'] == r['codigo_estacao']) &
            diario['data'].between(r['data_inicio'], r['data_fim'])
        )
        n_removidas = mask.sum()
        diario = diario[~mask]
        print(f"Removidas {n_removidas} linha(s) de {r['codigo_estacao']} "
              f"({r['data_inicio']} a {r['data_fim']}) -- motivo: {r['motivo']}")
else:
    print(f"Nenhum {caminho_exclusoes.name} encontrado -- rode o 01_eda_horaria.ipynb "
          "primeiro se quiser aplicar exclusões manuais. Seguindo sem excluir nada.")

print("\nShape final (diário), depois das exclusões:", diario.shape)
print(diario['codigo_estacao'].nunique(), "estações únicas")
print("Período coberto:", diario['data'].min(), "a", diario['data'].max())


# ============================================================
# CÉLULA 5 — Salvar o diário
# ============================================================
caminho_saida = PASTA_PROCESSADA / "clima_diario_agg.parquet"
diario.to_parquet(caminho_saida, index=False)
print(f"Salvo em {caminho_saida}")

c:\Users\okubo\OneDrive\Documentos\PCD2\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


9288 arquivos encontrados em c:\Users\okubo\OneDrive\Documentos\PCD2\data\raw\DadosClimaticos


Processando: 100%|██████████| 9288/9288 [06:29<00:00, 23.85it/s]



0 arquivo(s) com falha no processamento.

Shape final (diário), antes das exclusões manuais: (2765762, 17)
Nenhum exclusoes_manuais.csv encontrado -- rode o 01_eda_horaria.ipynb primeiro se quiser aplicar exclusões manuais. Seguindo sem excluir nada.

Shape final (diário), depois das exclusões: (2765762, 17)
614 estações únicas
Período coberto: 2005-01-01 00:00:00 a 2024-12-31 00:00:00
Salvo em c:\Users\okubo\OneDrive\Documentos\PCD2\data\processed\clima_diario_agg.parquet
